# Annual-Report-Agent (AI in Finance)

Eigene Version des FinRobot-Annual-Report-Agenten: Datenquellen SEC EDGAR + Yahoo Finance, automatische Peer-Suche, Quellenangaben [1]-[4], Segment- und Peer-Tabelle, automatischer Faktencheck im Anhang.

Ablauf: Zellen nacheinander ausfuehren. Firma, Geschaeftsjahr und Modell stellst du in Zelle 2 und 3 ein.

In [ ]:
# 1) Pfade und Imports
import os, sys, json, glob
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
FR = ROOT / "finrobot_autogen"
sys.path.insert(0, str(AIF))

import autogen
from finrobot.utils import register_keys_from_json
from lib.report_agent import build_agent, default_message
print("FinRobot-Ordner:", ROOT)

In [ ]:
# 2) Modell und Schluessel
MODEL = "anthropic/claude-haiku-5.5"   # muss in OAI_CONFIG_LIST stehen

llm_config = {
    "config_list": autogen.config_list_from_json(str(FR / "OAI_CONFIG_LIST"), filter_dict={"model": [MODEL]}),
    "timeout": 300,
    "temperature": 0.5,
    "parallel_tool_calls": False,   # Tools nacheinander (FinRobot verkraftet keine parallelen Aufrufe)
    "max_tokens": 16000,            # Obergrenze pro Antwort (Kostenkontrolle)
}
assert llm_config["config_list"], f"{MODEL} nicht in OAI_CONFIG_LIST gefunden"
register_keys_from_json(str(FR / "config_api_keys"))

In [ ]:
# 3) Firma, Geschaeftsjahr, Peers
COMPANY = "RKLB"     # Ticker
FYEAR = "2025"       # Geschaeftsjahr des 10-K (Periodenende)
PEERS = None         # None = Agent sucht Peers selbst (Finnhub + Yahoo); oder z. B. ["LUNR", "RDW", "PL"]

tag = MODEL.split("/")[-1]
work_dir = f"../reports/{COMPANY}_{FYEAR}_{tag}"   # relativer Pfad (vermeidet Backslash-Probleme im Prompt)
os.makedirs(work_dir, exist_ok=True)
message = default_message(COMPANY, FYEAR, work_dir, PEERS)
print(message)

In [ ]:
# 4) Agent starten
assistant = build_agent(llm_config, max_rounds=30)   # max. 30 automatische Runden (Schutz vor Endlosschleifen)
assistant.chat(message, use_cache=True, max_turns=60, summary_method="last_msg")

In [ ]:
# 5) Ergebnis anzeigen: Faktencheck-Zusammenfassung und PDF-Seiten
import io
import fitz
from PIL import Image

fcs = sorted(glob.glob(f"{work_dir}/*_factcheck.json"), key=os.path.getmtime)
if fcs:
    d = json.load(open(fcs[-1], encoding="utf-8"))
    print(f"Faktencheck: {d['matched']} von {d['total_claims']} Zahlen in den Quellen gefunden ({(d['match_rate'] or 0)*100:.0f}%)")
    print("Peers:", d.get("peer_tickers"))
    for i in d["name_issues"]:
        print(f"  Namensfehler: {i['written']} ({i['ticker']}) -> offiziell {i['official']}")

pdfs = sorted(glob.glob(f"{work_dir}/*.pdf"), key=os.path.getmtime)
if not pdfs:
    print("Kein PDF erzeugt - siehe Ausgabe von Zelle 4.")
else:
    print("PDF:", pdfs[-1])
    for page in fitz.open(pdfs[-1]):
        display(Image.open(io.BytesIO(page.get_pixmap(dpi=100).tobytes("png"))))